In [4]:
import os, sys

path_project = os.getcwd()
os.chdir(path_project)
sys.path.insert(0, path_project)

In [5]:
os.getcwd()

'/workspace/finetune/dwarf'

In [6]:
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace

from scripts.schema_converter import convert_schema_text

/root/miniconda3/envs/py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
engine = create_engine("sqlite:///database/code_snippet_management_and_evaluation/code_snippet_management_and_evaluation.sqlite")

In [8]:
database = SQLDatabase(
    engine, # connection to the database
    sample_rows_in_table_info=3 # number of rows to show in table info
)

In [ ]:
PROMPT_TEXT_TO_SQL = '''You are an expert SQL assistant. Use ONLY the provided table schema to write a correct SQL query. Return valid SQL without explanations.
Table schema:
{table_info}
Question: {input}
Output format enclose the generated SQL query in a code block::
```
-- Your SQL query
```
'''

prompt_template_text_to_sql = PromptTemplate(
    template=PROMPT_TEXT_TO_SQL,
    input_variables=["table_info", "input"],
)

In [10]:
def get_schema(_):
    table_info_original = database.get_table_info()
    table_info = convert_schema_text(table_info_original)
    return table_info

print("Schema:", get_schema({}))

Schema: CREATE TABLE code_snippets (
	snippet_id INTEGER, 
	description TEXT, 
	reference_code TEXT, 
	complexity INTEGER, 
	language_id INTEGER, 
	uploaded_by INTEGER, 
	upload_date TEXT, 
	last_modified TEXT, 
	version INTEGER, 
	is_public INTEGER, 
	is_archived INTEGER, 
	license TEXT, 
	file_path TEXT, 
	file_size INTEGER, 
	PRIMARY KEY (snippet_id), 
	CONSTRAINT fk_code_snippets_uploaded_by FOREIGN KEY(uploaded_by) REFERENCES users (user_id), 
	CONSTRAINT fk_code_snippets_language_id FOREIGN KEY(language_id) REFERENCES programming_languages (language_id)
);
INSERT INTO code_snippets (snippet_id, description, reference_code, complexity, language_id, uploaded_by, upload_date, last_modified, version, is_public, is_archived, license, file_path, file_size) VALUES (0, 'Multifactorial of n of order k, n(!!...!).', 'def factorialk(n, k, exact=True): \n if exact: ...', 5, 0, 1, '2023-01-01', '2023-01-01', 1, 1, 0, 'MIT', 'path/to/snippet1.py', 1024), (1, 'Issues an HTTP redirect to the giv

In [11]:
def get_response(query):
    response = database.run(query)
    return response

print("Response:", get_response("SELECT * FROM snippet_comments LIMIT 5"))

Response: [(0, 0, 0, 'Great function, works well for large numbers.', '2023-05-01', 0, '2023-05-01'), (1, 1, 0, 'Useful for web redirects.', '2023-05-02', 0, '2023-05-02')]


In [12]:
import re
def post_process_sql(sql):
    # get text after "<start_of_turn>model"
    pattern = r"<start_of_turn>model(.*)"
    match = re.search(pattern, sql, re.DOTALL)
    if match:
        sql = match.group(1).strip()
        # ambil text start select sampai sebelum ;
        pattern = r"(SELECT.*?);"
        match = re.search(pattern, sql, re.DOTALL | re.IGNORECASE)
        if match:
            sql = match.group(1).strip()
            return sql
    return None

In [14]:
model_path = os.path.join(os.getcwd(), "outputs/gemma-3-1b-sql-qlora/merged")

In [15]:
from langchain_huggingface.llms import HuggingFacePipeline

In [16]:
model = HuggingFacePipeline.from_model_id(
    model_id = model_path,
    task="text-generation",
    model_kwargs={"trust_remote_code": True, "device_map":"auto"},
)

Device set to use cuda:0


In [17]:
chat = ChatHuggingFace(
    llm=model,
    temperature=0,
)

In [18]:
sql_chain_response = (
    RunnablePassthrough.assign(
        table_info=get_schema,
    )
    | prompt_template_text_to_sql
    | chat
    | StrOutputParser()
)

def sql_generator(query: str) -> str:
    sql = sql_chain_response.invoke({"input": query})
    sql = post_process_sql(sql)
    return sql

In [19]:
sql_response = sql_generator("Could you please count the total number of comments for the code snippet with the id 1?")

In [20]:
print(sql_response)

SELECT COUNT(*) FROM snippet_comments sc JOIN code_snippets cs ON sc.snippet_id = cs.snippet_id WHERE cs.snippet_id = 1


In [21]:
database.run(sql_response)

'[(1,)]'